# Sparsity: Systematic Study

Top-k neuron sparsity. Dominates sort (+21pp) and mazes (+8pp).

| tier | what | runs |
|---|---|---|
| **Main** | baseline vs sparsity(0.5), 5 seeds x 2 tasks | 10 |
| **Sweep** | ratio {0.1..0.9}, 3 seeds x 2 tasks | 30 |
| **Total** | | **40** |

**Hardware**: 1 machine x 8 GPUs (~6h).

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Part A - Prior Results (st08 sparsity0.5)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    print(summary_stats(df_prior[(df_prior.stage=='st08') & (df_prior.sweep=='sparsity0.5')]))
else:
    print('Prior data not found.')

In [ ]:
if df_prior is not None:
    plot_prior_bar(df_prior, ['sort','mazes'],
                   'st08', 'sparsity0.5', 'Prior: sparsity 0.5 vs baseline',
                   'figures/03_prior_bar.png')

In [ ]:
curves = load_prior_curves()
if curves:
    plot_prior_curves(curves, 'sort',
        [('st00','paper','baseline','#888'),
         ('st08','sparsity0.5','sparsity 0.5','#9467bd')],
        'Sort convergence (prior)', 'figures/03_prior_conv.png')

## Part B - Experiment Design

In [ ]:
main_exps = make_sparsity(['sort','mazes'], [0,1,2,3,4], ratios=[0.5])
sweep_exps = make_sparsity_sweep(['sort','mazes'], [0,1,2])
exps = main_exps + sweep_exps
print(f'Main: {len(main_exps)}, Sweep: {len(sweep_exps)}')
print(f'Total: {len(exps)} experiments')

In [ ]:
run_all(exps[:5], gpus=8, log_root='logs/deep/03_sparsity', dry_run=True)
print('...')
run_all(exps[-5:], gpus=8, log_root='logs/deep/03_sparsity', dry_run=True)

### Run all (~6h on 8 GPUs)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/03_sparsity')

In [ ]:
status('logs/deep/03_sparsity')

## Part C - Main Results

In [ ]:
df = collect('logs/deep/03_sparsity')
if df.empty:
    print('No results yet.')
else:
    df_main = df[df.name.str.contains('sparsity0p5_s') & ~df.name.str.contains('swp')]
    if not df_main.empty:
        print(df_main[['name','task','best_acc','delta']].to_string(index=False))
        plot_delta_bars(df_main, 'Main: sparsity(0.5) vs baseline (5 seeds)',
                        'figures/03_main_delta.png')

In [ ]:
if not df.empty:
    df_main = df[df.name.str.contains('sparsity0p5_s') & ~df.name.str.contains('swp')]
    if not df_main.empty:
        sig = significance_test(df_main)
        print(sig.to_string(index=False))

In [ ]:
if not df.empty:
    df_main = df[df.name.str.contains('sparsity0p5_s') & ~df.name.str.contains('swp')]
    if not df_main.empty:
        plot_box_seeds(df_main, 'task', 'best_acc',
                       'Main: seed variance', 'figures/03_main_box.png')

## Part D - Ratio Sensitivity

In [ ]:
if not df.empty:
    import re
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        df_sw['ratio'] = df_sw['name'].str.extract(r'r([0-9]+p?[0-9]*)_s')[0].str.replace('p','.').astype(float)
        plot_sweep_heatmap(df_sw, 'ratio', 'task',
                          'Sparsity: ratio x task delta (pp)',
                          'figures/03_sweep_heatmap.png')

In [ ]:
if not df.empty:
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        import re
        df_sw['ratio'] = df_sw['name'].str.extract(r'r([0-9]+p?[0-9]*)_s')[0].str.replace('p','.').astype(float)
        plot_sweep_curve(df_sw, 'ratio',
                        'Sparsity ratio sweep (errorbar = std over seeds)',
                        'figures/03_sweep_curve.png')